# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 04d: Schedule Notebook Pipeline as Task DAG

**Objective:** Create a Task DAG that runs notebooks 04a, 04b, and 04c in sequence on a daily schedule.

```
TASK_NB_PIPELINE_ROOT (Daily 6 AM ET)
  └── TASK_NB_04A_STATEMENTS (AFTER ROOT)
        └── TASK_NB_04B_LATE_PAYMENTS (AFTER 04A)
              └── TASK_NB_04C_ACTIVITY_SNAPSHOT (AFTER 04B)
```

**Key Concept:** `EXECUTE NOTEBOOK` inside a Task allows you to schedule Snowflake Notebooks as part of automated pipelines - combining the interactivity of notebooks with production scheduling.

In [ ]:
-- Setup context
USE ROLE ACCOUNTADMIN;
USE DATABASE JPMC_CONSUMER_BANKING_HOL;
USE SCHEMA HOL_LAB;
USE WAREHOUSE HOL_MEDIUM_WH;

## Step 1: Create Sequential Task DAG for Notebook Pipeline

All three transformation notebooks run in sequence:
1. **04a - Monthly Statements** (runs first - generates fresh statement data)
2. **04b - Late Payment Flagging** (depends on statements from 04a)
3. **04c - Customer Activity Snapshot** (runs last - summarizes all activity)

In [ ]:
-- =============================================================================
-- TASK DAG: Sequential notebook execution pipeline
-- 04a → 04b → 04c (each waits for the previous to complete)
-- =============================================================================

-- Root task: triggers daily at 6 AM ET
CREATE OR REPLACE TASK TASK_NB_PIPELINE_ROOT
    WAREHOUSE = HOL_MEDIUM_WH
    SCHEDULE = 'USING CRON 0 6 * * * America/New_York'
AS
    SELECT 1;

-- Step 1: Monthly Statements
CREATE OR REPLACE TASK TASK_NB_04A_STATEMENTS
    WAREHOUSE = HOL_MEDIUM_WH
    AFTER TASK_NB_PIPELINE_ROOT
AS
    EXECUTE NOTEBOOK JPMC_CONSUMER_BANKING_HOL.HOL_LAB."04a_NB_Monthly_Statements";

-- Step 2: Late Payment Flagging (depends on fresh statements)
CREATE OR REPLACE TASK TASK_NB_04B_LATE_PAYMENTS
    WAREHOUSE = HOL_MEDIUM_WH
    AFTER TASK_NB_04A_STATEMENTS
AS
    EXECUTE NOTEBOOK JPMC_CONSUMER_BANKING_HOL.HOL_LAB."04b_NB_Late_Payment_Flagging";

-- Step 3: Customer Activity Snapshot (runs last)
CREATE OR REPLACE TASK TASK_NB_04C_ACTIVITY_SNAPSHOT
    WAREHOUSE = HOL_MEDIUM_WH
    AFTER TASK_NB_04B_LATE_PAYMENTS
AS
    EXECUTE NOTEBOOK JPMC_CONSUMER_BANKING_HOL.HOL_LAB."04c_NB_Customer_Activity_Snapshot";

## Step 2: Resume Tasks and Verify

Tasks are created in a suspended state. Resume them bottom-up (children first, root last) to activate the DAG.

In [ ]:
-- Resume tasks (bottom-up: children first, root last)
ALTER TASK TASK_NB_04C_ACTIVITY_SNAPSHOT RESUME;
ALTER TASK TASK_NB_04B_LATE_PAYMENTS RESUME;
ALTER TASK TASK_NB_04A_STATEMENTS RESUME;
ALTER TASK TASK_NB_PIPELINE_ROOT RESUME;

-- Verify task DAG
SHOW TASKS LIKE 'TASK_NB_%' IN SCHEMA HOL_LAB;

## Step 3: Test the Pipeline Manually

In [ ]:
-- Manually trigger the pipeline (doesn't wait for cron schedule)
EXECUTE TASK TASK_NB_PIPELINE_ROOT;

In [ ]:
-- Check execution status (wait ~30 seconds after EXECUTE TASK for results)
SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME, RETURN_VALUE
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    SCHEDULED_TIME_RANGE_START => DATEADD(HOUR, -1, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 10
))
WHERE NAME LIKE 'TASK_NB_%'
ORDER BY SCHEDULED_TIME DESC;